In [2]:
# Uncomment if packages are not yet installed in your local environment
!pip -q install cmdstanpy pyarrow pandas numpy

In [3]:
import os
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from cmdstanpy import CmdStanModel, install_cmdstan
from pathlib import Path

# ── Local paths (relative to this notebook's directory) ──────────────────────
NOTEBOOK_DIR = Path(__file__).parent if '__file__' in dir() else Path('.').resolve()

PARQUET_PATH = NOTEBOOK_DIR / "processing" / "processed_data" / "full_task_all.parquet"
STAN_PATH    = NOTEBOOK_DIR / "bayesian_model" / "main.stan"
VI_STAN_PATH = NOTEBOOK_DIR / "bayesian_model" / "VarInf.stan"

assert PARQUET_PATH.exists(), f"Parquet not found: {PARQUET_PATH}"
assert STAN_PATH.exists(),    f"Stan model not found: {STAN_PATH}"
assert VI_STAN_PATH.exists(), f"VI Stan model not found: {VI_STAN_PATH}"
print("Found files OK.")

Found files OK.


/Users/kshoker/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Install CmdStan only if it is not already installed
import cmdstanpy
try:
    cmdstanpy.cmdstan_path()  # raises RuntimeError if not installed
    print("CmdStan already installed at:", cmdstanpy.cmdstan_path())
except ValueError:
    install_cmdstan(version="2.35.0", cores=2)
    print("CmdStan installed.")

CmdStan already installed at: /Users/kshoker/.cmdstan/cmdstan-2.38.0


In [5]:
pf = pq.ParquetFile(str(PARQUET_PATH))
md = pf.metadata.metadata or {}

def md_get_str(key: str, default=None):
    b = md.get(key.encode("utf-8"))
    if b is None:
        return default
    try:
        return b.decode("utf-8")
    except Exception:
        return default

K_str = md_get_str("K")
vocab_str = md_get_str("skill_vocab_json")

print("Parquet metadata keys:", sorted([k.decode("utf-8", "ignore") for k in md.keys()])[:20], "...")
print("K metadata:", K_str)
print("Has skill_vocab_json:", vocab_str is not None)

K = int(K_str) if K_str is not None else None
if K is None:
    raise RuntimeError("Missing Parquet metadata 'K'. Your full_task_all.parquet may not be the right build.")

# optional: parse vocab for debugging
skill_vocab = None
if vocab_str:
    # processing/main.ipynb stored `str(dict)` not strict JSON, so parse carefully
    # We'll try JSON first, then eval fallback.
    try:
        skill_vocab = json.loads(vocab_str)
    except Exception:
        try:
            skill_vocab = eval(vocab_str, {"__builtins__": {}})
        except Exception:
            skill_vocab = None

print("K =", K)
print("skill_vocab parsed:", isinstance(skill_vocab, dict))

Parquet metadata keys: ['ARROW:schema', 'K', 'skill_vocab_json'] ...
K metadata: 69
Has skill_vocab_json: True
K = 69
skill_vocab parsed: True


In [6]:
df_raw = pd.read_parquet(PARQUET_PATH)

# Expect these columns from processing/main.ipynb
expected = {"QuestionId","UserId","AnswerId","IsCorrect","Gender","TimeTaken","Age","skill_indices"}
missing = expected - set(df_raw.columns)
if missing:
    raise RuntimeError(f"Missing expected columns: {missing}. Columns are: {list(df_raw.columns)[:50]}")

# Filter TimeTaken similarly to main.r
df = df_raw.copy()
df = df[df["TimeTaken"].notna()]
df = df[(df["TimeTaken"] >= 5) & (df["TimeTaken"] <= 1800)].copy()

df["log_time"] = np.log(df["TimeTaken"].astype(float))
df["is_fast"] = (df["TimeTaken"] < 20).astype(np.int32)

print("Rows after filter:", len(df))
df.head()

Rows after filter: 877716


,QuestionId,UserId,AnswerId,IsCorrect,Gender,TimeTaken,Age,skill_indices,log_time,is_fast
1522,26942,63580,8712723,1,1,240.0,14.863792,[48],5.480639,0
2047,16742,60749,17684380,1,2,1320.0,14.483231,[49],7.185387,0
2827,4437,37877,3026515,1,1,300.0,14.362765,[22],5.703782,0
2870,7955,28905,10251452,1,0,60.0,NaN,[58],4.094345,0
3823,24527,118109,5684672,1,0,60.0,NaN,[7],4.094345,0


In [ ]:
# ── Student filtering for HMC feasibility ─────────────────────────────────────
# Apply AFTER TimeTaken filtering.
MIN_ANSWERS_PER_STUDENT = 30
N_STUDENTS = 500
RANDOM_SEED = 42

counts = df.groupby("UserId").size()
eligible_users = counts[counts >= MIN_ANSWERS_PER_STUDENT].index

print("Eligible students (>=", MIN_ANSWERS_PER_STUDENT, "answers after TimeTaken filter):", len(eligible_users))

if len(eligible_users) < N_STUDENTS:
    raise ValueError(
        f"Only {len(eligible_users)} eligible students, cannot sample N_STUDENTS={N_STUDENTS}. "
        "Lower N_STUDENTS or MIN_ANSWERS_PER_STUDENT."
    )

rng = np.random.default_rng(RANDOM_SEED)
selected_users = rng.choice(eligible_users.to_numpy(), size=N_STUDENTS, replace=False)

# Filter main modeling dataframe to selected students
df = df[df["UserId"].isin(selected_users)].copy()

# Sanity checks
print("Rows after student filter:", len(df))
print("Unique students after filter:", df["UserId"].nunique())

post_counts = df.groupby("UserId").size()
print(
    "Answers per student after filter (min/median/mean/max):",
    int(post_counts.min()),
    float(post_counts.median()),
    float(post_counts.mean()),
    int(post_counts.max()),
)

Eligible students (>= 20 answers after TimeTaken filter): 5518
Rows after student filter: 148626
Unique students after filter: 5000
Answers per student after filter (min/median/mean/max): 20 25.0 29.7252 218


In [ ]:
# ── Row sampling (after student filter) ─────────────────────────────────────
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

N_TARGET = 10000  # start small; try 200000 if it runs
ANCHOR_PER_GROUP = 5

# anchors: up to 5 per (UserId, IsCorrect)
anchors = (
    df.groupby(["UserId", "IsCorrect"], group_keys=False)
      .apply(lambda g: g.sample(n=min(len(g), ANCHOR_PER_GROUP), random_state=RANDOM_SEED))
      .reset_index(drop=True)
)

remaining = df[~df["AnswerId"].isin(anchors["AnswerId"])]

need = max(0, N_TARGET - len(anchors))
top_up = remaining.sample(n=min(need, len(remaining)), random_state=RANDOM_SEED)

sample_df = pd.concat([anchors, top_up], ignore_index=True)
sample_df = sample_df.sample(n=min(N_TARGET, len(sample_df)), random_state=RANDOM_SEED).reset_index(drop=True)

print("Sample rows:", len(sample_df))
print("Unique users (should be <= 5000):", sample_df["UserId"].nunique())
print("Unique questions:", sample_df["QuestionId"].nunique())

# hard guard
assert sample_df["UserId"].nunique() <= 5000

/var/folders/yj/3yq26dgs2nn76q3v6zwh32mc0000gn/T/ipykernel_73023/1553867668.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), ANCHOR_PER_GROUP), random_state=RANDOM_SEED))


Sample rows: 50000
Unique users (should be <= 5000): 5000
Unique questions: 13400


In [9]:
sample_df["player_idx"] = pd.factorize(sample_df["UserId"])[0].astype(np.int32) + 1
sample_df["question_idx"] = pd.factorize(sample_df["QuestionId"])[0].astype(np.int32) + 1

n_players = int(sample_df["player_idx"].max())
n_questions = int(sample_df["question_idx"].max())

assert n_players == sample_df["player_idx"].nunique()
assert n_questions == sample_df["question_idx"].nunique()

print("n_players:", n_players)
print("n_questions:", n_questions)

n_players: 5000
n_questions: 13400


In [10]:
# Build mapping from original QuestionId to remapped question_idx
# NOTE: skill_indices entries are list-like (often numpy arrays) and unhashable,
# so we must NOT call drop_duplicates() on [QuestionId, skill_indices].

q_idx_map = (
    sample_df[["QuestionId", "question_idx"]]
    .drop_duplicates(subset=["QuestionId"])
)

q_skill_map = (
    df_raw[["QuestionId", "skill_indices"]]
    .drop_duplicates(subset=["QuestionId"])
)

q_map = (
    q_idx_map
    .merge(q_skill_map, on="QuestionId", how="left")
    .sort_values("question_idx")
    .reset_index(drop=True)
)

assert len(q_map) == n_questions
assert q_map["question_idx"].is_unique

# Ensure each skill_indices is a list[int]
def normalize_skill_list(x):
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if isinstance(x, (list, tuple)):
        return [int(v) for v in x]
    try:
        return [int(v) for v in list(x)]
    except Exception:
        return []

skills_per_q = [normalize_skill_list(x) for x in q_map["skill_indices"].tolist()]

# bounds check to [1..K]
skills_per_q = [[s for s in sl if 1 <= s <= K] for sl in skills_per_q]

skill_count = np.array([len(sl) for sl in skills_per_q], dtype=np.int32)
skill_start = np.ones(n_questions, dtype=np.int32)
if n_questions > 1:
    skill_start[1:] = (np.cumsum(skill_count)[:-1] + 1).astype(np.int32)

skill_flat = np.array([s for sl in skills_per_q for s in sl], dtype=np.int32)
n_skill_entries = int(len(skill_flat))

# sanity
assert np.all(skill_count >= 0)
assert np.all(skill_start >= 1)
assert int(skill_count.sum()) == n_skill_entries

print("n_skill_entries:", n_skill_entries)
print("Skill sparsity:", round(1 - n_skill_entries / (n_questions * K), 3))

n_skill_entries: 14082
Skill sparsity: 0.985


In [11]:
player_meta = (
    sample_df[["player_idx","Gender","Age"]]
    .drop_duplicates(subset=["player_idx"])
    .sort_values("player_idx")
    .reset_index(drop=True)
)

assert len(player_meta) == n_players

age_raw = player_meta["Age"].astype(float).to_numpy()
age_mean = np.nanmean(age_raw)
age_sd = np.nanstd(age_raw)
if not np.isfinite(age_sd) or age_sd == 0:
    age_sd = 1.0

age_z = np.where(np.isfinite(age_raw), (age_raw - age_mean) / age_sd, 0.0).astype(np.float64)

gender = player_meta["Gender"].to_numpy()
# match Stan: 0 unknown, 1 ref, 2 other
gender = np.where(pd.isna(gender), 0, gender).astype(np.int32)
gender = np.clip(gender, 0, 2)

print("age_z shape:", age_z.shape)
print("gender counts:", pd.Series(gender).value_counts().to_dict())

age_z shape: (5000,)
gender counts: {2: 1985, 1: 1806, 0: 1209}


In [12]:
stan_data = {
    "N": int(len(sample_df)),
    "n_players": n_players,
    "n_questions": n_questions,
    "n_skills": int(K),

    "player_id": sample_df["player_idx"].astype(np.int32).to_numpy(),
    "question_id": sample_df["question_idx"].astype(np.int32).to_numpy(),
    "response": sample_df["IsCorrect"].astype(np.int32).to_numpy(),
    "is_fast": sample_df["is_fast"].astype(np.int32).to_numpy(),
    "log_time": sample_df["log_time"].astype(np.float64).to_numpy(),

    "n_skill_entries": int(n_skill_entries),
    "skill_flat": skill_flat.astype(np.int32),
    "skill_start": skill_start.astype(np.int32),
    "skill_count": skill_count.astype(np.int32),

    "age_z": age_z.astype(np.float64),
    "gender": gender.astype(np.int32),
}

# hard checks like main.r
assert stan_data["player_id"].max() == stan_data["n_players"]
assert stan_data["question_id"].max() == stan_data["n_questions"]
assert len(stan_data["age_z"]) == stan_data["n_players"]
assert len(stan_data["gender"]) == stan_data["n_players"]
assert len(stan_data["skill_start"]) == stan_data["n_questions"]
assert len(stan_data["skill_count"]) == stan_data["n_questions"]
print("stan_data assembled OK.")

stan_data assembled OK.


In [13]:
# Stan files are local and already writable — compile directly from source.
STAN_LOCAL_PATH = STAN_PATH
VI_LOCAL_PATH   = VI_STAN_PATH

model = CmdStanModel(stan_file=str(STAN_LOCAL_PATH))
print("Compiled main.stan OK.")

13:12:51 - cmdstanpy - INFO - compiling stan file /var/folders/yj/3yq26dgs2nn76q3v6zwh32mc0000gn/T/tmp14rak2ip/tmpsus5cpxn.stan to exe file /Users/kshoker/Desktop/STAT 405/project/bayesian_model/main
13:13:02 - cmdstanpy - INFO - compiled model executable: /Users/kshoker/Desktop/STAT 405/project/bayesian_model/main


Compiled main.stan OK.


In [ ]:
fit = model.sample(
    data=stan_data,
    chains=2,
    parallel_chains=2,
    iter_warmup=300,
    iter_sampling=300,
    show_progress=True,
)

13:13:06 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 1/600 [00:01<12:20,  1.24s/it, (Warmup)]13:16:32 - cmdstanpy - ERROR - Chain [2] error: code '-2' Unknown error: -2
13:16:32 - cmdstanpy - ERROR - Chain [1] error: code '-2' Unknown error: -2


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import os

# Create a safe directory for your outputs
output_dir = "/kaggle/working/bayesian_results"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. THE ULTIMATE BACKUP (Raw Stan CSVs)
# ---------------------------------------------------------
# This copies the gigabytes of raw MCMC data from the hidden /tmp/ folder 
# to your visible Kaggle working directory. If everything else fails, 
# you can rebuild your fit from these files.
fit.save_csvfiles(dir=output_dir)

# ---------------------------------------------------------
# 2. THE DIAGNOSTICS TABLE (For the Appendix & Convergence Check)
# ---------------------------------------------------------
summary_df = fit.summary()
summary_df.to_csv(f"{output_dir}/nuts_mcmc_summary.csv")

# ---------------------------------------------------------
# 3. LOG-LIKELIHOOD DRAWS (For LOO-CV & Model Comparison)
# ---------------------------------------------------------
# You MUST have this to compare your Complex Model to your Naive Model
log_lik_df = fit.draws_pd(vars=["log_lik"])
log_lik_df.to_csv(f"{output_dir}/log_lik_draws.csv", index=False)

# ---------------------------------------------------------
# 4. POSTERIOR PREDICTIVE CHECKS (y_rep)
# ---------------------------------------------------------
# You did the mean calculation, but you might want to plot a histogram 
# of the accuracy distributions for your "Critical Evaluation" section.
y_rep_df = fit.draws_pd(vars=["y_rep"])
y_rep_df.to_csv(f"{output_dir}/y_rep_draws.csv", index=False)

# ---------------------------------------------------------
# 5. TARGET STUDENT PROFILES (For the "Real World" Inference Task)
# ---------------------------------------------------------
# Grab the 69 skill posteriors for the first 3 students so you can 
# make the "Skill Profile" bar charts the rubric asks for.
# Get all user_skill draws then filter to first 3 students' columns
all_skill_df = fit.draws_pd(vars=["user_skill"])
student_cols = [c for c in all_skill_df.columns
                if any(c.startswith(f"user_skill[{i},") for i in range(1, 4))]
student_df = all_skill_df[student_cols]
student_df.to_csv(f"{output_dir}/target_student_profiles.csv", index=False)

print(f"All critical data saved successfully to {output_dir}!")

In [ ]:
# Extract a small set of draws for y_rep
draws = fit.draws_pd(vars=["y_rep"])
print("y_rep draws shape:", draws.shape)

obs_mean = float(np.mean(stan_data["response"]))
# draws_pd returns wide columns y_rep[1], y_rep[2], ...
yrep_cols = [c for c in draws.columns if c.startswith("y_rep[")]
pp_mean = float(draws[yrep_cols].to_numpy().mean())

print("Observed mean correct:", round(obs_mean, 4))
print("Posterior predictive mean correct (across draws/obs):", round(pp_mean, 4))

In [ ]:
from cmdstanpy import CmdStanModel

if VI_LOCAL_PATH is None:
    raise FileNotFoundError("VI_STAN_PATH not found (check filename/casing or dataset path).")

vi_model = CmdStanModel(stan_file=str(VI_LOCAL_PATH))
print("Compiled VarInf.stan OK.")

In [ ]:
vi_fit = vi_model.variational(
    data=stan_data,
    algorithm="meanfield",     # or "fullrank" (slower, more flexible)
    iter=20000,
    grad_samples=2,
    elbo_samples=100,
    draws=800,       # posterior draws from the variational approx
)
print("VI done.")

In [ ]:
# ---------------------------------------------------------
# 1. THE RAW VI CSV FILES (Backup)
# ---------------------------------------------------------
vi_fit.save_csvfiles(dir=output_dir)

# ---------------------------------------------------------
# 2. THE FULL VI DRAWS (For Density Plots)
# ---------------------------------------------------------
# Build draws DataFrame via stan_variable (vi_fit.sample removed in this cmdstanpy version)
global_vars = ["gamma_base", "alpha", "beta_time", "mu_log_time", "sigma_log_time"]
vi_draws_df = pd.DataFrame({v: vi_fit.stan_variable(v, mean=False) for v in global_vars})
vi_draws_df.to_csv(f"{output_dir}/vi_draws_full.csv", index=False)

# ---------------------------------------------------------
# 3. THE VI SUMMARY (For the Comparison Table)
# ---------------------------------------------------------
vi_summary_df = vi_draws_df.describe()
vi_summary_df.to_csv(f"{output_dir}/vi_summary_comparison.csv")

print(f"VI data successfully saved to {output_dir}!")


In [ ]:
import pandas as pd

global_vars = ["gamma_base", "alpha", "beta_time", "mu_log_time", "sigma_log_time"]
draws = pd.DataFrame({v: vi_fit.stan_variable(v, mean=False) for v in global_vars})
draws.describe()


In [ ]:
vi_fit.runset.csv_files  # list of output csv paths